# 1. Data Entry

For my data, I used [this](https://www.kaggle.com/datasets/mdabbert/ultimate-ufc-dataset) kaggle dataset from mdabbert. They were able to scrape [ufcstats.com](ufcstats.com) for fighter and bout information, [bestfightodds.com](bestfightodds.com) for closing odds, and [a separate kaggle dataset](https://www.kaggle.com/datasets/martj42/ufc-rankings) for historical ranking information. In this notebook, we are just going to look for missing data or potentially incorrect information.

In [1]:
# We don't need many imports for this step
import pandas as pd
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None) 

In [2]:
# Bringing in our csv data
raw_df = pd.read_csv("data/ufc-master.csv")

In [3]:
manualOddsPath = 'data/manual_red_blue_odds.csv'
oddsKeyColumns = ['Date', 'RedFighter', 'BlueFighter', 'WeightClass']
oddsColumns = oddsKeyColumns + ['RedOdds', 'BlueOdds']

# Bringing in manually entered odds to deal with missing data

manualOdds = pd.read_csv(manualOddsPath)
df = raw_df.copy()

# Replacing the scraper odds with the manually-entered odds wherever the fight keys match.
df = df.merge(manualOdds, on=oddsKeyColumns, how='left', suffixes=('', '_Manual'))
df['RedOdds'] = df['RedOdds_Manual'].combine_first(df['RedOdds'])
df['BlueOdds'] = df['BlueOdds_Manual'].combine_first(df['BlueOdds'])
df = df.drop(['RedOdds_Manual', 'BlueOdds_Manual'], axis=1)

print('Manual Odds Added', len(manualOdds))
print('Missing RedOdds after replacement:', df['RedOdds'].isna().sum())
print('Missing BlueOdds after replacement:', df['BlueOdds'].isna().sum())

Manual Odds Added 6383
Missing RedOdds after replacement: 0
Missing BlueOdds after replacement: 0


In [4]:
# Taking a look at all of our columns
colCount = 0
for each in df:
    colCount += 1
    print(str(colCount) + " " + each)

1 RedFighter
2 BlueFighter
3 RedOdds
4 BlueOdds
5 RedExpectedValue
6 BlueExpectedValue
7 Date
8 Location
9 Country
10 Winner
11 TitleBout
12 WeightClass
13 Gender
14 NumberOfRounds
15 BlueCurrentLoseStreak
16 BlueCurrentWinStreak
17 BlueDraws
18 BlueAvgSigStrLanded
19 BlueAvgSigStrPct
20 BlueAvgSubAtt
21 BlueAvgTDLanded
22 BlueAvgTDPct
23 BlueLongestWinStreak
24 BlueLosses
25 BlueTotalRoundsFought
26 BlueTotalTitleBouts
27 BlueWinsByDecisionMajority
28 BlueWinsByDecisionSplit
29 BlueWinsByDecisionUnanimous
30 BlueWinsByKO
31 BlueWinsBySubmission
32 BlueWinsByTKODoctorStoppage
33 BlueWins
34 BlueStance
35 BlueHeightCms
36 BlueReachCms
37 BlueWeightLbs
38 RedCurrentLoseStreak
39 RedCurrentWinStreak
40 RedDraws
41 RedAvgSigStrLanded
42 RedAvgSigStrPct
43 RedAvgSubAtt
44 RedAvgTDLanded
45 RedAvgTDPct
46 RedLongestWinStreak
47 RedLosses
48 RedTotalRoundsFought
49 RedTotalTitleBouts
50 RedWinsByDecisionMajority
51 RedWinsByDecisionSplit
52 RedWinsByDecisionUnanimous
53 RedWinsByKO
54 RedWins

# 2. Data Cleaning

The first thing I noticed here is how many different columns there are! The creators of this dataset did an amazing job getting everything that's available, I just don't think I want all of this. Overtraining the model with so many data types might confuse it. I'm going to choose some columns that I'd like to try training models with later on.

I'm going a few different ways here. I want some that describe the fighters as old or young, streaking or slumping, strikers or grapplers, seasoned or green, and even tall or short. At this point, I can see myself trying out a few different kinds of models and I want to set myself up for success later on. Removing those extra columns like decision count or arena location just makes our data look cleaner and easier to scrape for.

In [5]:
# Dropping the individual weightclass ranking columns so the missing values table is easier to read
# Dropping some other extra columns that clutter up our data
weightRankColumns = [col for col in df.columns if (col.endswith('Rank') and col not in ['BMatchWCRank', 'RMatchWCRank'])]
otherExtraColumns = ['BlueDecOdds', 'RedDecOdds', 'BSubOdds', 'RSubOdds', 'RKOOdds', 'BKOOdds', 'FinishDetails', 'Finish', 'FinishRound', 
                     'TotalFightTimeSecs', 'BlueDraws', 'RedDraws', 'Location', 'Country', 'RedExpectedValue', 'BlueExpectedValue', 
                    'BlueWinsByDecisionMajority', 'BlueWinsByDecisionSplit', 'BlueWinsByDecisionUnanimous',
                    'RedWinsByDecisionMajority', 'RedWinsByDecisionSplit', 'RedWinsByDecisionUnanimous',
                    'BlueWeightLbs', 'RedWeightLbs', 'EmptyArena', 'FinishRoundTime',
                    'BlueAvgTDLanded', 'BlueAvgTDPct', 'BlueAvgSigStrPct', 'BlueAvgSubAtt', 'BlueAvgSigStrLanded',
                    'RedAvgTDLanded', 'RedAvgTDPct', 'RedAvgSigStrPct', 'RedAvgSubAtt', 'RedAvgSigStrLanded', 
                    'BlueWinsByTKODoctorStoppage', 'RedWinsByTKODoctorStoppage']
df = df.drop(weightRankColumns, axis=1, errors='ignore')
df = df.drop(otherExtraColumns, axis=1, errors='ignore')


print('Dropped weightclass rank columns:', len(weightRankColumns) + len(otherExtraColumns))

Dropped weightclass rank columns: 65


In [6]:
# Creating a new table that displays how many missing cells we have and how much of the column that takes up 
missing = pd.concat([df.isnull().sum(), 100 * df.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending = False).head(20)

,count,%
BMatchWCRank,5328,81.617647
RMatchWCRank,4749,72.748162
BlueStance,3,0.045956
BlueFighter,0,0.000000
RedOdds,0,0.000000
Date,0,0.000000
Winner,0,0.000000
WeightClass,0,0.000000
TitleBout,0,0.000000
NumberOfRounds,0,0.000000


As we can see, there are not many columns we need with data missing. There are a lot rank columns missing data because a lot of fighters are not ranked. Let's fill that in and then look at the stance data.

In [7]:
# Filling the Unranked fighters' rank cells with 20
df[['BMatchWCRank', 'RMatchWCRank']] = df[['BMatchWCRank', 'RMatchWCRank']].fillna(20)

# Checking whose stance information is missing
nullCheck = df[df['BlueStance'].isna()][['BlueFighter', 'BlueStance']]
nullCheck.head(10)

,BlueFighter,BlueStance
388,Daria Zhelezniakova,NaN
1817,Juancamilo Ronderos,NaN
1868,Juan Espino,NaN


In [8]:
# A quick search helped me fill this in
df.loc[df['BlueFighter'] == 'Daria Zhelezniakova', 'BlueStance'] = 'Orthodox'
df.loc[df['BlueFighter'] == 'Juancamilo Ronderos', 'BlueStance'] = 'Southpaw'
df.loc[df['BlueFighter'] == 'Juan Espino', 'BlueStance'] = 'Orthodox'

# Fixing an issue that came up during EDA
df.loc[df['BlueFighter'] == 'Eduardo Garagorri', 'BlueReachCms'] = 173.0

nullCheck = df[df['BlueStance'].isna()][['BlueFighter', 'BlueStance']]
nullCheck.head(10)

,BlueFighter,BlueStance


In [9]:
# Changing the date dtype
df['Date'] = pd.to_datetime(df['Date'])
print(df['Date'].dtype)

datetime64[us]


Above we inputted the missing fighters' stances and dropped rows where finishing information wasn't included. I put 20 as the fill rank for unranked fighters to show that a fighter unranked in a weightclass would typically be below a ranked fighter.

I, like others, believe that MMA has evolved a lot over the years. There is fight data in the original dataset that doesn't include rankings not because the fighters weren't ranked, but because rankings weren't introduced until 2013. I may want to remove some fights eventually, but for the sake of having a more complete database, I won't drop any yet.

In [10]:
#Final missing data check
missing = pd.concat([df.isnull().sum(), 100 * df.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending = False).head()

,count,%
RedFighter,0,0.0
BlueFighter,0,0.0
RedOdds,0,0.0
BlueOdds,0,0.0
Date,0,0.0


In [11]:
# Checking how many fights we have
print(len(df))

6528


In [12]:
# Looking at our data
df.head()

,RedFighter,BlueFighter,RedOdds,BlueOdds,Date,Winner,TitleBout,WeightClass,Gender,NumberOfRounds,BlueCurrentLoseStreak,BlueCurrentWinStreak,BlueLongestWinStreak,BlueLosses,BlueTotalRoundsFought,BlueTotalTitleBouts,BlueWinsByKO,BlueWinsBySubmission,BlueWins,BlueStance,BlueHeightCms,BlueReachCms,RedCurrentLoseStreak,RedCurrentWinStreak,RedLongestWinStreak,RedLosses,RedTotalRoundsFought,RedTotalTitleBouts,RedWinsByKO,RedWinsBySubmission,RedWins,RedStance,RedHeightCms,RedReachCms,RedAge,BlueAge,LoseStreakDif,WinStreakDif,LongestWinStreakDif,WinDif,LossDif,TotalRoundDif,TotalTitleBoutDif,KODif,SubDif,HeightDif,ReachDif,AgeDif,SigStrDif,AvgSubAttDif,AvgTDDif,BMatchWCRank,RMatchWCRank
0,Alexandre Pantoja,Kai Asakura,-250.0,215.0,2024-12-07,Red,True,Flyweight,MALE,5,0,0,0,0,0,0,0,0,0,Orthodox,172.72,175.26,0,6,6,3,42,3,2,4,12,Orthodox,165.10,170.18,34,31,0,-6,-6,-12,-3,-42,-3,-2,-4,7.62,5.08,-3,-4.41,-0.8,-2.61,20.0,0.0
1,Shavkat Rakhmonov,Ian Machado Garry,-210.0,295.0,2024-12-07,Red,False,Welterweight,MALE,3,0,8,8,0,20,0,3,0,8,Orthodox,190.50,187.96,0,6,6,0,11,0,1,5,6,Orthodox,185.42,195.58,30,27,0,2,2,2,0,9,0,2,-5,5.08,-7.62,-3,1.38,-1.5,-0.72,7.0,3.0
2,Ciryl Gane,Alexander Volkov,-380.0,300.0,2024-12-07,Red,False,Heavyweight,MALE,3,0,4,4,4,44,0,6,1,12,Orthodox,200.66,203.20,0,1,7,2,33,3,4,2,9,Orthodox,193.04,205.74,34,36,0,3,-3,3,2,11,-3,2,-1,7.62,-2.54,2,-0.36,-0.3,-0.13,3.0,2.0
3,Bryce Mitchell,Kron Gracie,-950.0,625.0,2024-12-07,Red,False,Featherweight,MALE,3,2,0,1,2,7,0,0,1,1,Southpaw,175.26,177.80,1,0,6,2,22,0,0,1,7,Southpaw,177.80,177.80,30,36,1,0,-5,-6,0,-15,0,0,0,-2.54,0.00,6,1.44,-1.1,-2.98,20.0,13.0
4,Nate Landwehr,Dooho Choi,-130.0,110.0,2024-12-07,Blue,False,Featherweight,MALE,3,0,1,3,3,15,0,4,0,4,Orthodox,177.80,177.80,0,1,3,3,17,0,1,2,5,Orthodox,175.26,182.88,36,33,0,0,0,-1,0,-2,0,3,-2,2.54,-5.08,-3,-1.84,-0.2,-0.25,20.0,20.0


Just taking a look at the data here, I actually want to clean it up a little bit. We have some floats where we don't need them and I'd like the objects that say corner colors to be boolean to help the computer out with the predictions.

In [13]:
# Changing some floats to integers
to_int = ['BMatchWCRank', 'RMatchWCRank']
df[to_int] = df[to_int].astype('int64')

# Making our winner column boolean to make things easier for the model
df['RedWinner'] = df['Winner'].map({'Red': True, 'Blue': False}).astype('bool')
df = df.drop('Winner', axis=1)
df = df.drop('BetterRank', axis=1, errors='ignore')

# Double checking
df.dtypes

RedFighter                          str
BlueFighter                         str
RedOdds                         float64
BlueOdds                        float64
Date                     datetime64[us]
TitleBout                          bool
WeightClass                         str
Gender                              str
NumberOfRounds                    int64
BlueCurrentLoseStreak             int64
BlueCurrentWinStreak              int64
BlueLongestWinStreak              int64
BlueLosses                        int64
BlueTotalRoundsFought             int64
BlueTotalTitleBouts               int64
BlueWinsByKO                      int64
BlueWinsBySubmission              int64
BlueWins                          int64
BlueStance                          str
BlueHeightCms                   float64
BlueReachCms                    float64
RedCurrentLoseStreak              int64
RedCurrentWinStreak               int64
RedLongestWinStreak               int64
RedLosses                         int64


Now that I removed the Winner column and instead made it RedWinner, we are going to be looking for results on a True/False basis instead of a Red/Blue basis. Since I have this, I want to add a couple of columns that deal specifically with the return for each bet. It will be a scale you multiply the amount of money bet, and it will show you how much your bet would return. 

I also decided a little bit later that it would be beneficial to have some data with the difference between the two fighters' features for the model to look at rather than just each isolated fighter's features. I'll be adding a few of those here.

In [14]:
# Defining my function
def calculate_return(odds, won):
    if not won:
        return 0
    if odds < 0:
        return round(1 + (-100 / odds), 4)
    else:
        return round(1 + (odds / 100), 4)

# Creating the new columns with the funtion appropriately applied    
df['RedReturn'] = df.apply(lambda row: calculate_return(row['RedOdds'], row['RedWinner']), axis=1)
df['BlueReturn'] = df.apply(lambda row: calculate_return(row['BlueOdds'], not row['RedWinner']), axis=1)

# Creating the difference columns while keeping the original Red and Blue columns in the dataframe
df['OddsDiff'] = df['RedOdds'] - df['BlueOdds']
df['AgeDiff'] = df['RedAge'] - df['BlueAge']
df['ReachDiff'] = df['RedReachCms'] - df['BlueReachCms']
df['HeightDiff'] = df['RedHeightCms'] - df['BlueHeightCms']
df['WinsDiff'] = df['RedWins'] - df['BlueWins']
df['LossesDiff'] = df['RedLosses'] - df['BlueLosses']
df['RoundsDiff'] = df['RedTotalRoundsFought'] - df['BlueTotalRoundsFought']
df['TitleBoutDiff'] = df['RedTotalTitleBouts'] - df['BlueTotalTitleBouts']
df['KODiff'] = df['RedWinsByKO'] - df['BlueWinsByKO']
df['SubmissionDiff'] = df['RedWinsBySubmission'] - df['BlueWinsBySubmission']
df['WinStreakDiff'] = df['RedCurrentWinStreak'] - df['BlueCurrentWinStreak']
df['LoseStreakDiff'] = df['RedCurrentLoseStreak'] - df['BlueCurrentLoseStreak']
df['LongestWinStreakDiff'] = df['RedLongestWinStreak'] - df['BlueLongestWinStreak']
df['SigStrDiff'] = df['SigStrDif']
df['SubAttDiff'] = df['AvgSubAttDif']
df['TDDiff'] = df['AvgTDDif']
df['RankDiff'] = df['RMatchWCRank'] - df['BMatchWCRank']

# Dropping the old difference columns from the original csv so I only have one naming style in this notebook
oldDiffColumns = ['LoseStreakDif', 'WinStreakDif', 'LongestWinStreakDif', 'WinDif', 'LossDif', 'TotalRoundDif',
                  'TotalTitleBoutDif', 'KODif', 'SubDif', 'HeightDif', 'ReachDif', 'AgeDif', 'SigStrDif',
                  'AvgSubAttDif', 'AvgTDDif']

# Dropping the individual weightclass ranking columns but keeping the match-specific ranks I use for RankDiff
weightRankColumns = [col for col in df.columns if (col.endswith('Rank') and col not in ['BMatchWCRank', 'RMatchWCRank'])]
df = df.drop(oldDiffColumns + weightRankColumns, axis=1, errors='ignore')

# Putting my columns in a different order for better readability
diffColumns = ['OddsDiff','AgeDiff','ReachDiff','HeightDiff','WinsDiff','LossesDiff','RoundsDiff','TitleBoutDiff','KODiff',
               'SubmissionDiff','WinStreakDiff','LoseStreakDiff','LongestWinStreakDiff','SigStrDiff','SubAttDiff','TDDiff','RankDiff']
frontColumns = ['RedFighter','BlueFighter','RedOdds','BlueOdds','RedWinner','RedReturn','BlueReturn'] + diffColumns
frontColumns += ['Date','TitleBout','WeightClass','Gender','NumberOfRounds']
remainingColumns = [col for col in df.columns if col not in frontColumns]
df = df[frontColumns + remainingColumns]

df.head()

,RedFighter,BlueFighter,RedOdds,BlueOdds,RedWinner,RedReturn,BlueReturn,OddsDiff,AgeDiff,ReachDiff,HeightDiff,WinsDiff,LossesDiff,RoundsDiff,TitleBoutDiff,KODiff,SubmissionDiff,WinStreakDiff,LoseStreakDiff,LongestWinStreakDiff,SigStrDiff,SubAttDiff,TDDiff,RankDiff,Date,TitleBout,WeightClass,Gender,NumberOfRounds,BlueCurrentLoseStreak,BlueCurrentWinStreak,BlueLongestWinStreak,BlueLosses,BlueTotalRoundsFought,BlueTotalTitleBouts,BlueWinsByKO,BlueWinsBySubmission,BlueWins,BlueStance,BlueHeightCms,BlueReachCms,RedCurrentLoseStreak,RedCurrentWinStreak,RedLongestWinStreak,RedLosses,RedTotalRoundsFought,RedTotalTitleBouts,RedWinsByKO,RedWinsBySubmission,RedWins,RedStance,RedHeightCms,RedReachCms,RedAge,BlueAge,BMatchWCRank,RMatchWCRank
0,Alexandre Pantoja,Kai Asakura,-250.0,215.0,True,1.4000,0.0,-465.0,3,-5.08,-7.62,12,3,42,3,2,4,6,0,6,-4.41,-0.8,-2.61,-20,2024-12-07,True,Flyweight,MALE,5,0,0,0,0,0,0,0,0,0,Orthodox,172.72,175.26,0,6,6,3,42,3,2,4,12,Orthodox,165.10,170.18,34,31,20,0
1,Shavkat Rakhmonov,Ian Machado Garry,-210.0,295.0,True,1.4762,0.0,-505.0,3,7.62,-5.08,-2,0,-9,0,-2,5,-2,0,-2,1.38,-1.5,-0.72,-4,2024-12-07,False,Welterweight,MALE,3,0,8,8,0,20,0,3,0,8,Orthodox,190.50,187.96,0,6,6,0,11,0,1,5,6,Orthodox,185.42,195.58,30,27,7,3
2,Ciryl Gane,Alexander Volkov,-380.0,300.0,True,1.2632,0.0,-680.0,-2,2.54,-7.62,-3,-2,-11,3,-2,1,-3,0,3,-0.36,-0.3,-0.13,-1,2024-12-07,False,Heavyweight,MALE,3,0,4,4,4,44,0,6,1,12,Orthodox,200.66,203.20,0,1,7,2,33,3,4,2,9,Orthodox,193.04,205.74,34,36,3,2
3,Bryce Mitchell,Kron Gracie,-950.0,625.0,True,1.1053,0.0,-1575.0,-6,0.00,2.54,6,0,15,0,0,0,0,-1,5,1.44,-1.1,-2.98,-7,2024-12-07,False,Featherweight,MALE,3,2,0,1,2,7,0,0,1,1,Southpaw,175.26,177.80,1,0,6,2,22,0,0,1,7,Southpaw,177.80,177.80,30,36,20,13
4,Nate Landwehr,Dooho Choi,-130.0,110.0,False,0.0000,2.1,-240.0,3,5.08,-2.54,1,0,2,0,-3,2,0,0,0,-1.84,-0.2,-0.25,0,2024-12-07,False,Featherweight,MALE,3,0,1,3,3,15,0,4,0,4,Orthodox,177.80,177.80,0,1,3,3,17,0,1,2,5,Orthodox,175.26,182.88,36,33,20,20


There we go -- I think that will do fine. I decided rather in favor of saving as much data as I can now and filtering it out on the modeling step. A more complete database will require more code and later on in the project, but will also prevent backtracking.

# 3. Data Validation

Now we have an accurate and informative 6000+ fight dataset with to train help train our model. While all of this looks good, let's check for some basic inconsistencies in the data. The first one I want to check is the age column. I want to make sure that we are seeing the age of the fighter at the time of the fight and not the same age all the way down. To check this, I will look at all of Dustin Poirier's UFC fights and see how he's aged.

In [15]:
# Age Check
mask = (df['RedFighter'] == 'Dustin Poirier') | (df['BlueFighter'] == 'Dustin Poirier')
df[mask][['RedFighter', 'RedAge', 'Date', 'BlueFighter', 'BlueAge']]

,RedFighter,RedAge,Date,BlueFighter,BlueAge
278,Islam Makhachev,32,2024-06-01,Dustin Poirier,35
404,Dustin Poirier,35,2024-03-09,Benoit Saint Denis,28
693,Dustin Poirier,34,2023-07-29,Justin Gaethje,34
1055,Dustin Poirier,33,2022-11-12,Michael Chandler,36
1523,Charles Oliveira,32,2021-12-11,Dustin Poirier,32
1745,Dustin Poirier,32,2021-07-10,Conor McGregor,32
1970,Dustin Poirier,32,2021-01-23,Conor McGregor,32
2272,Dustin Poirier,31,2020-06-27,Dan Hooker,30
2609,Khabib Nurmagomedov,30,2019-09-07,Dustin Poirier,30
2825,Max Holloway,27,2019-04-13,Dustin Poirier,30


Age came out perfectly! We can see Dustin from age 21 in his 2011 debut to age 35 in the most recent fight in this dataset. Now I want to make sure that the win streaks work as intended. I'll try this out with a different fighter this time, a fighter I know put together a few win streaks but also lost.

In [16]:
# Win Streak Check
mask = (df['RedFighter'] == 'Justin Gaethje') | (df['BlueFighter'] == 'Justin Gaethje')
df[mask][['RedFighter', 'RedCurrentWinStreak', 'RedLongestWinStreak', 'RedWinner', 'Date', 'BlueFighter', 'BlueCurrentWinStreak', 'BlueLongestWinStreak']]

,RedFighter,RedCurrentWinStreak,RedLongestWinStreak,RedWinner,Date,BlueFighter,BlueCurrentWinStreak,BlueLongestWinStreak
342,Justin Gaethje,2,4,False,2024-04-13,Max Holloway,2,13
693,Dustin Poirier,1,5,False,2023-07-29,Justin Gaethje,1,4
895,Justin Gaethje,0,4,True,2023-03-18,Rafael Fiziev,6,6
1326,Charles Oliveira,10,10,True,2022-05-07,Justin Gaethje,1,4
1576,Justin Gaethje,0,4,True,2021-11-06,Michael Chandler,0,3
2085,Khabib Nurmagomedov,12,12,True,2020-10-24,Justin Gaethje,4,4
2356,Tony Ferguson,12,12,False,2020-05-09,Justin Gaethje,3,3
2598,Donald Cerrone,0,8,False,2019-09-14,Justin Gaethje,2,2
2838,Edson Barboza,1,4,False,2019-03-30,Justin Gaethje,1,1
3134,Justin Gaethje,0,1,True,2018-08-25,James Vick,4,5


As we can see, Justin Gaethje wins his first few fights to build his current and longest streak, but then loses to Khabib Nurmagomedov to reset it. He had built back up a 2 fight winning streak at the time of his Max Holloway fight, but the longest streak remained unaffected.

I want to now look at ranking information. I'll look at this using Israel Adesanya -- a fighter who fought through the rankings and became middleweight champion, but also took a light-heavyweight fight during that reign. We want to see Israel Adesanya's rank rise to champion but be reset to unranked during the Jan Blachowicz fight, going back to champion until some variance around the Sean Strickland and Alex Pereira fights. While we're here, we can also look at the title fight indicator. Let's see what this looks like.

In [17]:
# Rankings Check
mask = (df['RedFighter'] == 'Israel Adesanya') | (df['BlueFighter'] == 'Israel Adesanya')
df[mask][['TitleBout', 'RedFighter', 'RMatchWCRank', 'RedWinner', 'Date', 'WeightClass', 'BlueFighter', 'BMatchWCRank']]

,TitleBout,RedFighter,RMatchWCRank,RedWinner,Date,WeightClass,BlueFighter,BMatchWCRank
158,True,Dricus Du Plessis,0,True,2024-08-17,Middleweight,Israel Adesanya,2
620,True,Israel Adesanya,0,False,2023-09-09,Middleweight,Sean Strickland,5
872,True,Alex Pereira,0,False,2023-04-08,Middleweight,Israel Adesanya,1
1053,True,Israel Adesanya,0,False,2022-11-12,Middleweight,Alex Pereira,4
1243,True,Israel Adesanya,0,True,2022-07-02,Middleweight,Jared Cannonier,2
1462,True,Israel Adesanya,0,True,2022-02-12,Middleweight,Robert Whittaker,1
1781,True,Israel Adesanya,0,True,2021-06-12,Middleweight,Marvin Vettori,3
1917,True,Jan Blachowicz,0,True,2021-03-06,Light Heavyweight,Israel Adesanya,20
2130,True,Israel Adesanya,0,True,2020-09-26,Middleweight,Paulo Costa,2
2372,True,Israel Adesanya,0,True,2020-03-07,Middleweight,Yoel Romero,3


At first glance this may look weird, but this data is actually correct. We can see that in all of the title bouts, the red fighter's rank is 0.0, indicating championship. Israel Adesanya fights to his eventual championship, defends twice, fights the light-heavyweight champion and loses, then moves down to middleweight for a few more championship fights where he was or wasn't the champion.

The final thing I want to check here is wins by submission or knockout. Fans all know that Charles Oliveira is the best fighter to test this with since consistently finishes fights by both method.

In [18]:
# Finish Check
mask = (df['RedFighter'] == 'Charles Oliveira') | (df['BlueFighter'] == 'Charles Oliveira')
df[mask][['RedWinner', 'RedFighter', 'RedWinsByKO', 'RedWinsBySubmission', 'Date', 'BlueFighter','BlueWinsByKO', 'BlueWinsBySubmission']]

,RedWinner,RedFighter,RedWinsByKO,RedWinsBySubmission,Date,BlueFighter,BlueWinsByKO,BlueWinsBySubmission
28,True,Charles Oliveira,4,16,2024-11-16,Michael Chandler,3,1
343,False,Charles Oliveira,4,16,2024-04-13,Arman Tsarukyan,4,0
780,True,Charles Oliveira,3,16,2023-06-10,Beneil Dariush,3,5
1089,False,Charles Oliveira,3,16,2022-10-22,Islam Makhachev,2,5
1326,True,Charles Oliveira,3,15,2022-05-07,Justin Gaethje,5,0
1523,True,Charles Oliveira,3,14,2021-12-11,Dustin Poirier,11,3
1819,True,Charles Oliveira,2,14,2021-05-15,Michael Chandler,2,1
2016,False,Tony Ferguson,0,0,2020-12-12,Charles Oliveira,0,0
2361,False,Kevin Lee,3,4,2020-03-14,Charles Oliveira,2,13
2492,True,Charles Oliveira,1,13,2019-11-16,Jared Gordon,1,0


Perfect. We can see Charles Oliveira's submission and knockout totals climb as he wins his fights. This database looks ready to train with.

In [19]:
df.to_csv('data/testing.csv', index = False)